In [385]:
from qdrant_client import QdrantClient
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter
from pathlib import Path
import numpy as np
import torch
import gc
import re

In [386]:
COLLECT_NAME = "wb"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
CHUNK_SIZE = [num for num in range(150, 3000, 100)] # в токенах
CHUNK_OVERLAP = 75

In [387]:
len(CHUNK_SIZE)

29

In [388]:
# local_data 
local_data = Path.cwd() / "data"

# Проверка памяти GPU
def check_gpu_memory():
    """Выводит информацию о памяти GPU"""
    if not torch.cuda.is_available():
        print("GPU не доступен")
        return

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    total = torch.cuda.get_device_properties(0).total_memory / 1024**2

    print(f"📊 GPU Memory Status:")
    print(f"   Total: {total:.0f} MB")
    print(f"   Allocated: {allocated:.1f} MB ({allocated/total*100:.1f}%)")
    print(f"   Reserved: {reserved:.1f} MB ({reserved/total*100:.1f}%)")
    print(f"   Free: {total - reserved:.1f} MB")


# Очистка памяти перед загрузкой новой модели
def cleanup_memory():
    """Полная очистка памяти GPU"""
    # Удаляем все большие объекты
    for obj in gc.get_objects():
        try:
          if isinstance(obj, torch.Tensor):
              del obj
        except:
          pass

    # Очищаем кэш
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Сборщик мусора
    gc.collect()

    print("✅ Память очищена")
    check_gpu_memory()


def drop_embed_model_from_gpu():
    Settings.embed_model = None
    cleanup_memory()

In [389]:
reader = SimpleDirectoryReader(input_dir=local_data, required_exts=[".pdf"])
docs = reader.load_data()
print(len(docs))
assert len(docs) == 86, "ОШИБКА, должно быть 86"

# full_text = "\n\n".join([d.text for d in docs])
# all_docs = Document(text=full_text)

86


In [390]:
drop = []
for i, doc in enumerate(docs):
    doc.metadata["page_label"] = i + 1
    if len(doc.text) == 0:
        print(len(doc.text))
        drop.append(i-1)

0
0


In [391]:
drop

[32, 84]

In [392]:
for d in drop:
    docs.pop(d)

In [393]:
len(docs)

84

In [394]:
docs

[Document(id_='65fd8517-7c56-4787-863b-d5a9edd6b43b', embedding=None, metadata={'page_label': 1, 'file_name': 'Оферта.pdf', 'file_path': '/home/yuri/HSE/ВКР/RAG/data/Оферта.pdf', 'file_type': 'application/pdf', 'file_size': 385626, 'creation_date': '2026-03-28', 'last_modified_date': '2026-03-28'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='ОФЕРТАоб оказании услуг в ПВЗ\n1. Термины и определения\n1.1. Активация ПВЗ — процесс, в результате которого ПВЗ Исполнителя становится действующим, моментом\nАктивации ПВЗ признается получение Исполнителем подтверждения от Заказчика о возможной Активации в\nсоответствии с п. 3.2 Договора.\n1.2. Ау

In [395]:
full_text = []

for i, d in enumerate(docs):
    marker = f"<<<PAGE:{i+1}>>>"
    full_text.append(f"\n\n{marker}\n{d.text}\n{marker}\n\n")

full_text = "\n\n".join(full_text)

In [396]:
all_docs = Document(text=full_text)

In [397]:
len(all_docs.text)

253400

In [398]:
def enrich_nodes_with_page_info(nodes):
    """Propagating + надёжный парсинг"""
    current_pages = ["1"]   # на случай, если первый маркер не сразу попал
    
    for node in nodes:
        # Ищем маркер в любом виде (самый устойчивый вариант)
        found = re.findall(r'<<<PAGE:(\d+)>>>', node.text)
        
        if found:
            current_pages = sorted(set(found))   # обновляем текущую страницу
        
        # Присваиваем чанку страницы
        node.metadata["pages_covered"] = current_pages[:]
        
        # Полностью удаляем ВСЕ маркеры из текста
        cleaned = re.sub(r'<<<PAGE:\d+>>>', '', node.text)
        node.text = cleaned.strip()
    
    return nodes

In [399]:
def get_chunking(chunk_size, chunk_overlap, docs):
    # делим документы на чанки Sentence сплиттером
    # splitter = SentenceSplitter(chunk_size=chunk_size,
    #                             chunk_overlap=chunk_overlap)
    # делим на чанки Token сплиттером
    splitter = TokenTextSplitter(chunk_size=chunk_size, 
                                 chunk_overlap=chunk_overlap)
    nodes = splitter.get_nodes_from_documents([docs])
    # print(nodes[728])
    nodes = enrich_nodes_with_page_info(nodes)
    # print(nodes[728])

    return nodes

# nodes = get_chunking(CHUNK_SIZE[0], CHUNK_OVERLAP, all_docs)


In [400]:
def create_collect(chunk_size, docs):
    # загрузка embedding модели
    # Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
    
    client = QdrantClient(url="http://127.0.0.1:6333")

    col_name = f"{COLLECT_NAME}_token_spl_{chunk_size}_{CHUNK_OVERLAP}"

    if client.collection_exists(collection_name=col_name):
        client.delete_collection(collection_name=col_name)

    # инициализируем векторное хранилище
    vector_store = QdrantVectorStore(collection_name=col_name,
                                     client=client)
    # создаём индекс
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    
    # загружаем документы
    nodes = get_chunking(chunk_size, CHUNK_OVERLAP, docs)
    

    # создаём индекс документов
    index = VectorStoreIndex(
        nodes=nodes,
        storage_context=storage_context,
        show_progress=True 
    )
    # пытаемся удалить embedding модель с GPU
    # drop_embed_model_from_gpu()


    print(f"Коллекция {col_name} успешно создана и проиндексирована")
    # return client, index

# create_collect()

In [401]:
# for col in client.get_collections().collections:
#     name = col.name
#     client.delete_collection(name)

In [402]:
drop_embed_model_from_gpu()

Embeddings have been explicitly disabled. Using MockEmbedding.


/tmp/ipykernel_54793/1471043076.py:28: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, torch.Tensor):


✅ Память очищена
📊 GPU Memory Status:
   Total: 15832 MB
   Allocated: 8.1 MB (0.1%)
   Reserved: 20.0 MB (0.1%)
   Free: 15811.6 MB


In [ ]:
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

for chunk_size in CHUNK_SIZE:
    create_collect(chunk_size, all_docs)

In [406]:
drop_embed_model_from_gpu()

Embeddings have been explicitly disabled. Using MockEmbedding.


/tmp/ipykernel_54793/1471043076.py:28: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, torch.Tensor):


✅ Память очищена
📊 GPU Memory Status:
   Total: 15832 MB
   Allocated: 8.1 MB (0.1%)
   Reserved: 20.0 MB (0.1%)
   Free: 15811.6 MB
